In [1]:
from functions import *

import numpy as np
import os
from scipy.signal import butter, filtfilt, find_peaks
from scipy.linalg import logm
from scipy.io import savemat, loadmat
import scipy.io as spio
import pandas as pd
from datetime import datetime
from vedo import Points, Plotter, Line, Grid
from IPython.display import Video
import imageio.v2 as imageio
import shutil
import seaborn as sns
import stumpy

import matplotlib
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error

from mobgap.data import GenericMobilisedDataset
from mobgap.pipeline import MobilisedPipelineImpaired
from mobgap.aggregation import get_mobilised_dmo_thresholds


import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 
warnings.filterwarnings("ignore", category=UserWarning)

C:\Users\ac4jmi\Desktop\DMO4LNC\dmo4lnc-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ac4jmi\Desktop\DMO4LNC\dmo4lnc-analysis\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\ac4jmi\Desktop\DMO4LNC\dmo4lnc-analysis\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please re

In [6]:
# should videos be generated?
generate_videos = False
prints = False
regenerate_data_files = False

base_path = r"C:\Users\ac4jmi\Desktop\DMO4LNC\dmo4lnc-analysis\Dataset"

# Loop through all the participant folders syncing axivity sensors and mocap timings
for cohort in os.listdir(base_path):
    cohort_path = os.path.join(base_path, cohort)
    if os.path.isdir(cohort_path):
        for participant in os.listdir(cohort_path):
            participant_path = os.path.join(cohort_path, participant)
            if os.path.isdir(participant_path):
                participant_path = os.path.join(participant_path, 'Lab\\')
                print(participant_path)  # full path to each session folder
                if (not os.path.exists(os.path.join(participant_path, 'Laboratory\\data.mat'))) or (regenerate_data_files):
                    generate_trimmed_resampled_axivity(participant_path, prints = prints)
                    align_imu_mocap(participant_path, prints = prints)
                    if generate_videos:
                        video_path = os.path.join(participant_path, 'Motion Capture Data')
                        video_path = os.path.join(video_path, '')
                        tsv_files = [
                            os.path.splitext(f)[0]
                            for f in os.listdir(video_path)
                            if f.endswith('.tsv') and not f.startswith('synced')
                        ]
                        
                        for file_name in tsv_files:
                            generate_video_from_raw_data(file_name, video_path, prints = False)
                            Video(video_path+file_name+".mp4", embed=True, width=608)
                else:
                    print("data.mat already exists for this file!")

print("Preprocessing Done!")

C:\Users\ac4jmi\Desktop\DMO4LNC\dmo4lnc-analysis\Dataset\CP\234\Lab\
data.mat already exists for this file!
C:\Users\ac4jmi\Desktop\DMO4LNC\dmo4lnc-analysis\Dataset\CP\383\Lab\
data.mat already exists for this file!
C:\Users\ac4jmi\Desktop\DMO4LNC\dmo4lnc-analysis\Dataset\CP\582\Lab\
data.mat already exists for this file!
C:\Users\ac4jmi\Desktop\DMO4LNC\dmo4lnc-analysis\Dataset\CP\718\Lab\
data.mat already exists for this file!
C:\Users\ac4jmi\Desktop\DMO4LNC\dmo4lnc-analysis\Dataset\CP\855\Lab\
data.mat already exists for this file!
C:\Users\ac4jmi\Desktop\DMO4LNC\dmo4lnc-analysis\Dataset\CP\875\Lab\
data.mat already exists for this file!
C:\Users\ac4jmi\Desktop\DMO4LNC\dmo4lnc-analysis\Dataset\CP\884\Lab\
data.mat already exists for this file!
C:\Users\ac4jmi\Desktop\DMO4LNC\dmo4lnc-analysis\Dataset\CP\921\Lab\
data.mat already exists for this file!
C:\Users\ac4jmi\Desktop\DMO4LNC\dmo4lnc-analysis\Dataset\CP\969\Lab\
data.mat already exists for this file!
C:\Users\ac4jmi\Desktop\DMO4

# Now, run the MATLAB script to obtain standards...

### Then continue:

# Calculating DMOs from MobGap and Motion Capture

In [ ]:
# figure size settings
fig_scaler = 10
matplotlib.rcParams.update({'font.size': 2.5*fig_scaler})

# get mobgap dataset
paths_list = get_paths_with_extension("data.mat",
                                      start_location=os.path.join(os.getcwd(), "Dataset"),
                                      folders_to_ignore=["Home"])

dataset = get_mobilised_dataset(paths_list,
                                parent_folders_as_metadata=["cohort", "subject_id", "location", "sensor"])
print(dataset)

cohorts = ["CP", "PSP"] # can add "HA"
all_tests = ["Test" + str(i) for i in range(1, 10)]
all_metrics = pd.DataFrame(columns=["cohort", "subject", "file_name", "wb_ID", "wb_TPs", "wb_FPs", "wb_FNs", "wb_TNs", "mobgap_ICs", "mocap_ICs", 
                                    "IC_TPs", "IC_FPs", "IC_FNs", "stride_length_mobgap", "stride_length_mocap", "cadence_mocap", "cadence_mobgap"])

# get the HA healthy thresholds and rename them to "CP" so that we can use them to do some basic thresholding on the data
ha_thresholds = get_mobilised_dmo_thresholds().xs("HA", level=1, drop_level=False)
new_index = pd.MultiIndex.from_tuples([(dmo, cohort) for dmo, _ in ha_thresholds.index for cohort in cohorts], names=ha_thresholds.index.names)
duplicated_values = pd.concat([ha_thresholds] * len(cohorts), axis=0).reset_index(drop=True)
multi_cohort_thresholds = pd.DataFrame(duplicated_values.values, index=new_index, columns=ha_thresholds.columns)

subjects_to_ignore = ['969', '921'] # 969 was heavily gait impaired, 921 was not actually cp

for cohort in cohorts:
    data = dataset.get_subset(cohort=cohort)
    subjects = list({row[1] for row in dataset.group_labels if row[0] == cohort})
    subjects = list(set(subjects) - set(subjects_to_ignore)) # remove problem subjects
    print(subjects)
    
    # overwrite by uncommenting:
    #subjects = ['234']
    
    for subject in subjects:
        start_location = os.path.join(os.getcwd(), "Dataset")
        dat_files = get_paths_with_extension(extension="data.mat", start_location=start_location, folders_to_ignore=["Home"])
    
        # convert mat file to a dictionary
        print(f"Subject {subject}")
        print("Reading and converting mat file...")
        index = next((i for i, path in enumerate(dat_files) if f"\\{subject}\\" in path), None) # get the index of the chosen subject
        mat = loadmat_fixed(dat_files[index])
        print()
        
        for test_to_compare in all_tests:
            # run pipeline on current test for current subject

            # init variables for monitoring performance of mobgap
            cur_sub_metrics = pd.Series(index=["cohort", "subject", "file_name", "wb_ID", "wb_TPs", "wb_FPs", "wb_FNs", "wb_TNs", "mobgap_ICs", "mocap_ICs", 
                                               "IC_TPs", "IC_FPs", "IC_FNs", "stride_length_mobgap", "stride_length_mocap", "cadence_mocap", "cadence_mobgap"], dtype = 'object')
            
            test = data.get_subset(Test=test_to_compare, subject_id = str(subject))[0]
            pipeline = MobilisedPipelineImpaired(dmo_thresholds=multi_cohort_thresholds) # thresholding set to HA equivalent
            pipeline = pipeline.safe_run(test) # pipeline now stores all the results for all wbs
        
            # mat file shortcuts to reduce variable sizes
            cur_mat_root = mat["data"]["TimeMeasure1"][test_to_compare]["Trial1"]
            cur_file_name = cur_mat_root["FileName"]
            
            # wb sizes
            mobgap_all_wbs = len(pipeline.raw_ic_list_['ic'].index.get_level_values(0))
            if not mobgap_all_wbs == 0: 
                wbs_in_current_test_mobgap = (max(pipeline.raw_ic_list_['ic'].index.get_level_values(0))+1)
            else:
                wbs_in_current_test_mobgap = 0
        
            if "Stereophoto" in cur_mat_root["Standards"]:
                wbs_in_current_test_mocap = len(cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"])
                if wbs_in_current_test_mocap >= 60: # magic number caused by matlab implementation. outside scope to delve into the matlab code
                    wbs_in_current_test_mocap = 1
            else:
                wbs_in_current_test_mocap = 0
            
            print()
            print("######################### " + cur_file_name + " #########################")
        
            # add the stuff for this test to the metrics series
            cur_sub_metrics["cohort"] = cohort
            cur_sub_metrics["subject"] = subject
            cur_sub_metrics["file_name"] = cur_file_name
            cur_sub_metrics[["wb_TPs", "wb_FPs", "wb_FNs", "wb_TNs", "mobgap_ICs", "mocap_ICs", "IC_TPs", "IC_FPs", "IC_FNs"]] = 0
            
            
            if test_to_compare in ["Test1"]: 
                # if there are walking bouts detected in standing, we have false positives, else true negatives
                if wbs_in_current_test_mobgap >= 1: 
                    cur_sub_metrics["wb_FPs"] += wbs_in_current_test_mobgap
                else:
                    cur_sub_metrics["wb_TNs"] += 1

                cur_sub_metrics = cur_sub_metrics.infer_objects() # automatically get dtypes
                all_metrics = pd.concat([all_metrics, cur_sub_metrics.to_frame().T], ignore_index=True)
                    
            else: 
                # for all other activities, process each wb
                mocap_timestamp = cur_mat_root["SU"]["LowerBack"]["Timestamp"]
                
                if (wbs_in_current_test_mocap == 1) and (wbs_in_current_test_mobgap == 1): 
                    cur_sub_metrics["wb_ID"] = 1
                    cur_sub_metrics["wb_TPs"] = 1
                    # get the indices of all IC events detected by mobgap and mocap
                    mocap_ic_events = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"]["InitialContact_Event"]
                    mocap_event_indices = [get_closest_index(mocap_timestamp, ic_event) for ic_event in mocap_ic_events]
                    mobgap_event_indices = pipeline.raw_ic_list_['ic'].values

                    # reduce the IC events to only true positives and plot them
                    reduced_mobgap_event_indices, mask, mobgap_mask, mocap_mask, metrics = match_closest_unique(mocap_event_indices, mobgap_event_indices)
                    #plot_subject_wbs(test, mocap_event_indices, mobgap_event_indices)
        
                    # store the IC metrics
                    cur_sub_metrics["mobgap_ICs"] = len(mobgap_event_indices)
                    cur_sub_metrics["mocap_ICs"] = len(mocap_event_indices)
                    for k, v in metrics.items():
                        cur_sub_metrics[k] = v
                    
                    # get the DMOs for the current trial
                    cur_sub_metrics["stride_length_mocap"] = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"]["AverageStrideLength"]
                    cur_sub_metrics["stride_length_mobgap"] = pipeline.per_wb_parameters_["stride_length_m"].values
                    cur_sub_metrics["cadence_mocap"] = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"]["AverageCadence"]
                    cur_sub_metrics["cadence_mobgap"] = pipeline.per_wb_parameters_["cadence_spm"].values
                    cur_sub_metrics["walking_speed_mocap"] = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"]["AverageSpeed"]
                    cur_sub_metrics["walking_speed_mobgap"] = pipeline.per_wb_parameters_["walking_speed_mps"].values[0] # for some reason this one needs a [0]

                    cur_sub_metrics = cur_sub_metrics.infer_objects() # automatically get dtypes
                    all_metrics = pd.concat([all_metrics, cur_sub_metrics.to_frame().T], ignore_index=True)
                    
                else:
                    # more than 1 CWB means we need to loop through them and treat them like seperate tests
                    # first, check if mobgap and mocap both detected the same number of walking bouts
                    if wbs_in_current_test_mocap == wbs_in_current_test_mobgap:
                        print(f"{wbs_in_current_test_mocap} wbs detected!")
                        
                        for wb in range(wbs_in_current_test_mocap):
                            print(f"WB: {wb}")
                            cur_sub_metrics["wb_ID"] = wb+1
                            cur_sub_metrics["wb_TPs"] = 1
                            
                            # get the indices of all IC events detected by mobgap and mocap for this walking bout
                            mocap_ic_events = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"][wb]["InitialContact_Event"]
                            mocap_event_indices = [get_closest_index(mocap_timestamp, ic_event) for ic_event in mocap_ic_events]
                            mobgap_event_indices = pipeline.raw_ic_list_['ic'].loc[wb].values
        
                            # reduce the IC events to only true positives and plot them
                            reduced_mobgap_event_indices, mask, mobgap_mask, mocap_mask, metrics = match_closest_unique(mocap_event_indices, mobgap_event_indices)
                            #plot_subject_wbs(test, mocap_event_indices, mobgap_event_indices)

                            # store the IC metrics
                            cur_sub_metrics["mobgap_ICs"] = len(mobgap_event_indices)
                            cur_sub_metrics["mocap_ICs"] = len(mocap_event_indices)
                            for k, v in metrics.items():
                                cur_sub_metrics[k] = v
                            
                            # get the DMOs for the current wb
                            cur_sub_metrics["stride_length_mocap"] = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"][wb]["AverageStrideLength"]
                            cur_sub_metrics["stride_length_mobgap"] = pipeline.per_wb_parameters_["stride_length_m"].loc[wb]
                            cur_sub_metrics["cadence_mocap"] = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"][wb]["AverageCadence"]
                            cur_sub_metrics["cadence_mobgap"] = pipeline.per_wb_parameters_["cadence_spm"].loc[wb]
                            cur_sub_metrics["walking_speed_mocap"] = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"][wb]["AverageSpeed"]
                            if not cur_sub_metrics["walking_speed_mocap"]:
                                cur_sub_metrics["walking_speed_mocap"] = np.nan # needed because sometimes this is just an empty list and causes errors
                            cur_sub_metrics["walking_speed_mobgap"] = pipeline.per_wb_parameters_["walking_speed_mps"].loc[wb]
                            
                            cur_sub_metrics = cur_sub_metrics.infer_objects() # automatically get dtypes
                            all_metrics = pd.concat([all_metrics, cur_sub_metrics.to_frame().T], ignore_index=True)
                            
                    else: 
                        print(f"{wbs_in_current_test_mocap=}")
                        print(f"{wbs_in_current_test_mobgap=}")
                        # mismatched wbs - report false positive/false negatives
                        if wbs_in_current_test_mocap > wbs_in_current_test_mobgap:
                            cur_sub_metrics["wb_FNs"] = (wbs_in_current_test_mocap - wbs_in_current_test_mobgap)
                        else:
                            cur_sub_metrics["wb_FPs"] = (wbs_in_current_test_mobgap - wbs_in_current_test_mocap)
                            
                        cur_sub_metrics = cur_sub_metrics.infer_objects() # automatically get dtypes
                        all_metrics = pd.concat([all_metrics, cur_sub_metrics.to_frame().T], ignore_index=True)

# convert NAN columns to floats
cols_with_nan = [c for c in all_metrics.columns if all_metrics[c].isna().any()]

# convert lists to single values
all_metrics[cols_with_nan] = all_metrics[cols_with_nan].map(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else (np.nan if isinstance(x, list) else x)
)

all_metrics[cols_with_nan] = all_metrics[cols_with_nan].astype(float)

# format to 1dp and print
pd.options.display.float_format = lambda x: f"{x:.1f}"
display(all_metrics)

# save all metrics as a csv so we don't have to rerun this all the time
all_metrics.to_csv("In-Lab Parameters\\all_metrics.csv")


# Calculate Errors and Results

In [ ]:
def bland_altman_plot(data1, data2, cohort, dot_size, title, fig_scaler=12, *args, **kwargs):
    data1 = np.asarray(data1)
    data2 = np.asarray(data2)
    n_samples = len(data1)

    mean = np.mean([data1, data2], axis=0)
    diff = data1 - data2
    md   = np.mean(diff)
    sd   = np.std(diff, axis=0)

    plt.figure(figsize=(fig_scaler, fig_scaler * 5 / 8))

    sns.scatterplot(x=mean, y=diff, hue=cohort, size=dot_size, sizes=(20, 200), *args, **kwargs)


    plt.axhline(md,           color='gray', linestyle='--')
    plt.axhline(md + 1.96*sd, color='gray', linestyle='--')
    plt.axhline(md - 1.96*sd, color='gray', linestyle='--')
    plt.title(title)
    plt.xlabel("Mean of mocap and mobgap")
    plt.ylabel("Diff between mocap and mobgap")
    plt.legend(bbox_to_anchor=(1.05, 1), prop = { "size": 18 }, loc='upper left')
    
    plt.tight_layout()
    plt.show()

def errors(mobgap_wb_dmo, mocap_wb_dmo):
    absolute_error = []
    relative_error = []
    for SS, RS in zip(mobgap_wb_dmo, mocap_wb_dmo):
        absolute_error.append(abs(SS-RS))
        relative_error = abs((SS-RS)/RS)*100
    mae = np.nanmean(absolute_error)
    upper_loa = mae + 1.96*np.nanstd(absolute_error)
    lower_loa = mae - 1.96*np.nanstd(absolute_error)
    return absolute_error, relative_error, mae, upper_loa, lower_loa

# wb-level stats stats
# calculate IC results into metrics dataframe
mask = all_metrics["IC_TPs"] > 0  # only keep rows where wb_ID is not NaN
all_metrics.loc[mask, "IC_Sensitivity"] = (
    all_metrics.loc[mask, "IC_TPs"] /
    (all_metrics.loc[mask, "IC_TPs"] + all_metrics.loc[mask, "IC_FNs"])
)
all_metrics.loc[mask, "IC_Positive_Predictive_Value"] = (
    all_metrics.loc[mask, "IC_TPs"] /
    (all_metrics.loc[mask, "IC_TPs"] + all_metrics.loc[mask, "IC_FPs"])
)
all_metrics.loc[mask, "IC_F1_Score"] = (
    (2*all_metrics.loc[mask, "IC_Sensitivity"]*all_metrics.loc[mask, "IC_Positive_Predictive_Value"]) /
    (all_metrics.loc[mask, "IC_Sensitivity"]+all_metrics.loc[mask, "IC_Positive_Predictive_Value"])
)

# subject-level stats
subject_metrics = pd.DataFrame(columns = ["Subject", "Cohort", "Accuracy", "Sensitivity", "Specificity", "Positive Predictive Value", "F1 Score"])
for subject in subjects:
    cur_sub_metrics = all_metrics[all_metrics["subject"] == subject]
    subject_metrics_ser = pd.Series(index=["Subject", "Cohort", "Accuracy", "Sensitivity", "Specificity", "Positive Predictive Value", "F1 Score"], dtype=object)
    subject_metrics_ser["Subject"] = subject
    subject_metrics_ser["Cohort"] = cur_sub_metrics["cohort"].iloc[0]
    subject_metrics_ser["Accuracy"] = (cur_sub_metrics["wb_TPs"].sum() + cur_sub_metrics["wb_TNs"].sum())/(cur_sub_metrics["wb_TPs"].sum() + cur_sub_metrics["wb_TNs"].sum() + cur_sub_metrics["wb_FPs"].sum() + cur_sub_metrics["wb_FNs"].sum())
    subject_metrics_ser["Sensitivity"] = (cur_sub_metrics["wb_TPs"].sum())/(cur_sub_metrics["wb_TPs"].sum() + cur_sub_metrics["wb_FNs"].sum())
    subject_metrics_ser["Positive Predictive Value"] = (cur_sub_metrics["wb_TPs"].sum())/(cur_sub_metrics["wb_TPs"].sum() + cur_sub_metrics["wb_FPs"].sum())
    subject_metrics_ser["Specificity"] = (cur_sub_metrics["wb_TNs"].sum())/(cur_sub_metrics["wb_TNs"].sum() + cur_sub_metrics["wb_FPs"].sum())
    subject_metrics_ser["F1 Score"] = (2*subject_metrics_ser["Positive Predictive Value"]*subject_metrics_ser["Sensitivity"])/(subject_metrics_ser["Positive Predictive Value"]+subject_metrics_ser["Sensitivity"])
    subject_metrics = pd.concat([subject_metrics, subject_metrics_ser.to_frame().T])


# cohort-level stats
cohort_metrics = pd.DataFrame(columns = ["Cohort", "Accuracy", "Sensitivity", "Specificity", "Positive Predictive Value", "F1 Score"])
cohorts = all_metrics["cohort"].unique()
for cohort in cohorts:
    cur_cohort_metrics = all_metrics[all_metrics["cohort"] == cohort]
    cohort_metrics_ser = pd.Series(index = ["Cohort", "Accuracy", "Sensitivity", "Specificity", "Positive Predictive Value", "F1 Score"], dtype=object)
    cohort_metrics_ser["Cohort"] = cohort
    cohort_metrics_ser["Accuracy"] = (cur_cohort_metrics["wb_TPs"].sum() + cur_cohort_metrics["wb_TNs"].sum())/(cur_cohort_metrics["wb_TPs"].sum() + cur_cohort_metrics["wb_TNs"].sum() + cur_cohort_metrics["wb_FPs"].sum() + cur_cohort_metrics["wb_FNs"].sum())
    cohort_metrics_ser["Sensitivity"] = (cur_cohort_metrics["wb_TPs"].sum())/(cur_cohort_metrics["wb_TPs"].sum() + cur_cohort_metrics["wb_FNs"].sum())
    cohort_metrics_ser["Positive Predictive Value"] = (cur_cohort_metrics["wb_TPs"].sum())/(cur_cohort_metrics["wb_TPs"].sum() + cur_cohort_metrics["wb_FPs"].sum())
    cohort_metrics_ser["Specificity"] = (cur_cohort_metrics["wb_TNs"].sum())/(cur_cohort_metrics["wb_TNs"].sum() + cur_cohort_metrics["wb_FPs"].sum())
    cohort_metrics_ser["F1 Score"] = (2*cohort_metrics_ser["Positive Predictive Value"]*cohort_metrics_ser["Sensitivity"])/(cohort_metrics_ser["Positive Predictive Value"]+cohort_metrics_ser["Sensitivity"])
    cohort_metrics = pd.concat([cohort_metrics, cohort_metrics_ser.to_frame().T])

# display all stats
display(all_metrics)
display(subject_metrics)
display(cohort_metrics)

all_metrics = all_metrics[all_metrics["cohort"] != "HA"] # drop HA
dot_size = pd.to_numeric(all_metrics.loc[mask, "IC_TPs"], errors='coerce').astype(float).values
bland_altman_plot(all_metrics.loc[mask, "stride_length_mocap"], all_metrics.loc[mask, "stride_length_mobgap"], all_metrics.loc[mask, "cohort"], dot_size, "Stride Length")
bland_altman_plot(all_metrics.loc[mask, "cadence_mocap"], all_metrics.loc[mask, "cadence_mobgap"], all_metrics.loc[mask, "cohort"], dot_size, "Cadence")
bland_altman_plot(all_metrics.loc[mask, "stride_length_mocap"], all_metrics.loc[mask, "stride_length_mobgap"], all_metrics.loc[mask, "subject"], dot_size, "Stride Length")
bland_altman_plot(all_metrics.loc[mask, "cadence_mocap"], all_metrics.loc[mask, "cadence_mobgap"], all_metrics.loc[mask, "subject"], dot_size, "Cadence")

print()
print(f"Mean IC Sensitivity = {all_metrics["IC_Sensitivity"].mean()}")
print(f"Mean IC Positive Predictive Value = {all_metrics["IC_Positive_Predictive_Value"].mean()}")
print(f"Mean IC F1 Score = {all_metrics["IC_F1_Score"].mean()}")
print()

print("####################### CP ########################")
_, _, mae_stride_length, upper_loa_stride_length, lower_loa_stride_length = errors(all_metrics[all_metrics["cohort"] == "CP"]["stride_length_mobgap"], all_metrics[all_metrics["cohort"] == "CP"]["stride_length_mocap"])
_, _, mae_cadence, upper_loa_cadence, lower_loa_cadence = errors(all_metrics[all_metrics["cohort"] == "CP"]["cadence_mobgap"], all_metrics[all_metrics["cohort"] == "CP"]["cadence_mocap"])
_, _, mae_walking_speed, upper_loa_walking_speed, lower_loa_walking_speed = errors(all_metrics[all_metrics["cohort"] == "CP"]["walking_speed_mobgap"], all_metrics[all_metrics["cohort"] == "CP"]["walking_speed_mocap"])

print()
print(f"MAE stride length = {mae_stride_length} [{lower_loa_stride_length}, {upper_loa_stride_length}]")
print(f"MAE cadence = {mae_cadence} [{lower_loa_cadence}, {upper_loa_cadence}]")
print(f"MAE walking speed = {mae_walking_speed} [{lower_loa_walking_speed}, {upper_loa_walking_speed}]")
#print(f"{std_absolute_error=}")